# 🌐 Notebook 1 — Data Exploration
**Project:** Uzbek Neural Machine Translation | **Author:** Asliddin

**Series:** Asliddin Builds #05

---
We explore the **OPUS-100** parallel corpus for Uzbek:
1. Load uz-en and uz-ru (via pivot) pairs from HuggingFace
2. Analyze sentence length distributions
3. Compare vocabulary richness across language pairs
4. Identify alignment quality issues
5. Build and save clean train/val/test splits for all 4 directions

In [ ]:
import sys
sys.path.append('../src')
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

plt.rcParams.update({'figure.dpi':120,'font.size':11})
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 1. Load OPUS-100 uz-en

`Helsinki-NLP/opus-100` config `en-uz` contains Uzbek-English parallel pairs.

In [ ]:
from datasets import load_dataset

try:
    ds_uz_en = load_dataset('Helsinki-NLP/opus-100', 'en-uz', split='train')
    print(f'OPUS-100 uz-en: {len(ds_uz_en):,} pairs')
    print('Sample:', ds_uz_en[0])
except Exception as e:
    print(f'[Demo] {e}')
    ds_uz_en = None

## 2. Sentence Length Distribution

In [ ]:
if ds_uz_en is not None:
    sample = ds_uz_en.select(range(min(5000, len(ds_uz_en))))
    uz_lens = [len(ex['translation']['uz'].split()) for ex in sample]
    en_lens = [len(ex['translation']['en'].split()) for ex in sample]
else:
    np.random.seed(42)
    uz_lens = np.random.gamma(3.5,4,5000).clip(1,80).tolist()
    en_lens = np.random.gamma(3.8,4,5000).clip(1,80).tolist()

fig, axes = plt.subplots(1,2,figsize=(13,5))
fig.suptitle('Sentence Length Distribution — OPUS-100 (Uzbek-English)', fontsize=13, fontweight='bold')

for ax, lens, lang, color in [
    (axes[0], uz_lens, 'Uzbek', '#66bb6a'),
    (axes[1], en_lens, 'English', '#4fc3f7')
]:
    ax.hist(lens, bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(np.median(lens), color='red', ls='--', lw=1.5, label=f'Median: {np.median(lens):.0f}')
    ax.axvline(128, color='orange', ls=':', lw=1.5, label='Max tokens (128)')
    ax.set_title(f'{lang} sentence lengths', fontweight='bold')
    ax.set_xlabel('Word count'); ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Uzbek — Mean: {np.mean(uz_lens):.1f} | Median: {np.median(uz_lens):.1f}')
print(f'English — Mean: {np.mean(en_lens):.1f} | Median: {np.median(en_lens):.1f}')

## 3. Length Ratio Analysis (alignment quality)

In [ ]:
ratios = [max(u,e)/max(min(u,e),1) for u,e in zip(uz_lens, en_lens)]

fig, ax = plt.subplots(figsize=(10,5))
ax.hist(ratios, bins=50, color='#ef5350', alpha=0.85, edgecolor='white')
ax.axvline(3.0, color='#ffd54f', ls='--', lw=2, label='Filter threshold (3.0x)')
ax.set_xlabel('Length ratio (longer/shorter)')
ax.set_ylabel('Count')
ax.set_title('Sentence Pair Length Ratio — OPUS-100 uz-en\n(ratio > 3.0 = likely misaligned, filtered out)', fontweight='bold')
ax.legend(fontsize=10)

filtered = sum(1 for r in ratios if r > 3.0)
print(f'Pairs with ratio > 3.0 (filtered): {filtered:,} ({filtered/len(ratios):.1%})')

plt.tight_layout()
plt.savefig('../results/alignment_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Prepare All 4 Directions

In [ ]:
from dataset import prepare_splits

for direction in ['uz-en','en-uz','uz-ru','ru-uz']:
    print(f'\nPreparing {direction}...')
    try:
        prepare_splits(direction, data_dir='../data/processed')
    except Exception as e:
        print(f'  Skipped: {e}')

print('\nAll directions prepared! → Next: Notebook 02 — MarianMT baseline')